In [1]:
!pip install rdkit

In [2]:
!pip install mordred

In [3]:
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

import pandas as pd
import numpy as np
from tqdm import tqdm
import os

In [4]:
dataset=pd.read_excel('/content/cluster1.xlsx')

In [5]:
df=dataset.dropna(subset=['smiles'])

In [6]:
df

,Compounds Name,IMPPAT_id,smiles
0,Hecogenin,IMPHY012275,C[C@@H]1CC[C@@]2(OC1)O[C@@H]1[C@H]([C@@H]2C)[C...
1,Kaempferol,IMPHY004388,Oc1ccc(cc1)c1oc2cc(O)cc(c2c(=O)c1O)O
2,Quercetin,IMPHY004619,Oc1cc(O)c2c(c1)oc(c(c2=O)O)c1ccc(c(c1)O)O
3,Ursolic acid,IMPHY011880,C[C@@H]1CC[C@]2([C@@H]([C@H]1C)C1=CC[C@H]3[C@@...
4,beta-Amyrin,IMPHY012223,O[C@H]1CC[C@]2([C@H](C1(C)C)CC[C@@]1([C@@H]2CC...
...,...,...,...
156,Myristic acid,IMPHY000060,CCCCCCCCCCCCCC(=O)O
157,Allyl isothiocyanate,IMPHY006321,C=CCN=C=S
158,Caffeine,IMPHY002965,Cn1cnc2c1c(=O)n(C)c(=O)n2C
159,"1,4-Diaminobutane",IMPHY000125,NCCCCN


In [7]:
def RDkit_descriptors(smiles):
    mols = [Chem.MolFromSmiles(i) for i in smiles]
    calc = MoleculeDescriptors.MolecularDescriptorCalculator([x[0] for x in Descriptors._descList])
    desc_names = calc.GetDescriptorNames()

    Mol_descriptors = []
    for mol in mols:
        # add hydrogens to molecules
        mol = Chem.AddHs(mol)
        # Calculate all 200 descriptors for each molecule
        descriptors = calc.CalcDescriptors(mol)
        Mol_descriptors.append(descriptors)
    return Mol_descriptors, desc_names

# Split the SMILES into chunks of 100,000 for faster processing
chunk_size = 100000
chunks = [dataset[i:i+chunk_size] for i in range(0, len(dataset), chunk_size)]

total_chunks=len(chunks)
total_time=0

# Check if there is an existing output file
if os.path.isfile('RDkit_descriptors.csv'):
    existing_data = pd.read_excel('RDkit_descriptors.csv', index_col=0)
else:
    existing_data = pd.DataFrame()
# Calculate descriptors for each chunk and concatenate the results
for i, chunk in enumerate(tqdm(chunks, desc='Processing', total=len(chunks))):
    # Check if this chunk has already been processed
    if len(existing_data) >= len(chunk):
        continue
    # Calculate descriptors for this chunk
    descriptors, desc_names = RDkit_descriptors(chunk['smiles'])
    # Convert the descriptors to a dataframe
    df_with_200_descriptors = pd.DataFrame(descriptors, columns=desc_names,)
    # Add the chunk index as a new column
    df_with_200_descriptors['chunk_index'] = i
    # Append the data to the existing data
    existing_data = pd.concat([existing_data, df_with_200_descriptors], axis=0)
    # Save the data after each chunk
    existing_data.to_csv('RDkit_descriptors.csv')

# Save the final data
existing_data.to_csv('RDkit_descriptors.csv')

Processing: 100%|██████████| 1/1 [00:04<00:00,  4.69s/it]


In [8]:
import pandas as pd
input_file = '/content/RDkit_descriptors.csv'

!pip install scikit-learn
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
data = pd.read_csv(input_file)
data

,Unnamed: 0,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,chunk_index
0,0,16.156013,16.156013,3.775249,-6.166692,0.589276,120.064516,430.629,388.293,430.308310,...,0,0,0,0,0,0,0,0,0,0
1,1,12.959421,12.959421,0.645293,-1.217861,0.545851,16.761905,286.239,276.159,286.047738,...,0,0,0,0,0,0,0,0,0,0
2,2,13.007550,13.007550,0.684876,-1.239332,0.434190,16.363636,302.238,292.158,302.042653,...,0,0,0,0,0,0,0,0,0,0
3,3,14.923825,14.923825,3.564375,-6.870025,0.414109,106.666667,456.711,408.327,456.360345,...,0,0,0,0,0,0,0,0,0,0
4,4,10.704594,10.704594,3.530522,-6.840009,0.387199,108.258065,426.729,376.329,426.386166,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,156,11.648008,11.648008,2.473122,-4.987832,0.488351,47.750000,228.376,200.152,228.208930,...,0,0,0,0,0,0,0,0,0,0
157,157,6.948009,6.948009,0.824213,-2.388310,0.288357,19.666667,99.158,94.118,99.014270,...,0,0,0,0,0,0,0,0,0,0
158,158,12.367750,12.367750,0.086921,-3.359734,0.538463,24.285714,194.194,184.114,194.080376,...,0,0,0,0,0,0,0,0,0,0
159,159,7.312708,7.312708,0.723074,-3.659375,0.464599,49.000000,88.154,76.058,88.100048,...,0,0,0,0,0,0,0,0,0,0


In [9]:
if 'Unnamed: 0' in data.columns:
    data = data.drop(columns=['Unnamed: 0'])

In [10]:
def discretize_column(col, bins=3):
    """Discretize a numerical column into equal-width bins."""
    return pd.cut(col, bins=bins, labels=False)

In [11]:
discretized_data = data.copy()
for column in discretized_data.select_dtypes(include=['float64', 'int64']).columns:
    discretized_data[column] = discretize_column(discretized_data[column])

In [12]:
encoder = OneHotEncoder(sparse_output=False)
encoded_data = encoder.fit_transform(discretized_data)

In [13]:
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(discretized_data.columns))

In [14]:
output_file = "onehot_encoded_rdkit_descriptors.csv"
encoded_df.to_csv(output_file, index=False)

print(f"One-hot encoded data saved to {output_file}")

One-hot encoded data saved to onehot_encoded_rdkit_descriptors.csv
